# Rollout to MP4

Run rollouts from robomimic checkpoints and save as mp4 videos.
Uses the latest trained diffusion policy checkpoint by default.

In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ.setdefault("MUJOCO_EGL_DEVICE_ID", "0")

REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

import imageio
import numpy as np
import torch
from IPython.display import Video, display

from src.latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint,
    build_rollout_policy_from_checkpoint,
    build_env_from_checkpoint,
)
from src.latent_sope.robomimic_interface.rollout import (
    rollout,
    get_policy_frame_stack,
)

## Configuration

In [ ]:
NUM_ROLLOUTS = 3
HORIZON = 400
CAMERAS = ["agentview"]
VIDEO_SKIP = 2
FPS = 20
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

## Find latest checkpoint and build policy + env

In [ ]:
# Auto-discover the latest training run
base = REPO_ROOT / "third_party" / "robomimic"
candidates = []
for trained_dir in base.glob("*_trained_models"):
    for experiment_dir in trained_dir.iterdir():
        if not experiment_dir.is_dir():
            continue
        for run_dir in experiment_dir.iterdir():
            if run_dir.is_dir() and (run_dir / "last.pth").exists():
                candidates.append(run_dir)
assert candidates, "No training runs found. Train a policy first (see hello_robomimic.ipynb Section 1.3)."
run_dir = sorted(candidates)[-1]
print(f"Run directory: {run_dir}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = load_checkpoint(run_dir, ckpt_path="last.pth")
print(f"Checkpoint: {ckpt.ckpt_path.name} (algo: {ckpt.algo_name})")

policy = build_rollout_policy_from_checkpoint(ckpt, device=device, verbose=False)
env = build_env_from_checkpoint(ckpt, render=False, render_offscreen=True, verbose=False)

frame_stack = get_policy_frame_stack(policy)
print(f"Policy frame_stack={frame_stack}, horizon={HORIZON}, cameras={CAMERAS}")

## Run rollouts and save videos

In [ ]:
out_dir = run_dir / "rollout_videos"
out_dir.mkdir(parents=True, exist_ok=True)

results = []
for i in range(NUM_ROLLOUTS):
    video_path = out_dir / f"rollout_{i:03d}.mp4"
    video_writer = imageio.get_writer(str(video_path), fps=FPS)

    stats = rollout(
        policy=policy,
        env=env,
        horizon=HORIZON,
        render=False,
        video_writer=video_writer,
        video_skip=VIDEO_SKIP,
        camera_names=CAMERAS,
    )
    video_writer.close()
    results.append((video_path, stats))

    status = "SUCCESS" if stats.success_rate > 0 else "FAIL"
    print(f"  [{i+1}/{NUM_ROLLOUTS}] {status}  reward={stats.total_reward:.2f}  "
          f"steps={stats.horizon}  -> {video_path.name}")

successes = sum(1 for _, s in results if s.success_rate > 0)
avg_reward = np.mean([s.total_reward for _, s in results])
print(f"\nDone: {successes}/{NUM_ROLLOUTS} successful, avg reward={avg_reward:.2f}")
print(f"Videos saved to: {out_dir}")

## Play back videos

In [ ]:
for video_path, stats in results:
    status = "SUCCESS" if stats.success_rate > 0 else "FAIL"
    print(f"{video_path.name}: {status}, reward={stats.total_reward:.2f}, steps={stats.horizon}")
    display(Video(str(video_path), embed=True, width=512))